# MidiLM: Conversational Music AI

Model AI musik berbasis MIDI dengan kemampuan percakapan teks.

In [ ]:
# 1. Install Library
!apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg
!pip install "numpy>=1.26.0,<2.0.0" "tensorflow>=2.16.1" "librosa>=0.10.0" "resampy>=0.4.0" --upgrade
!pip install "basic-pitch>=0.4.0" --no-deps
!pip install pretty_midi midi2audio torch tqdm mir_eval gguf tokenizers

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/MidiLM

In [ ]:
# 3. Prepare Dataset (Chat + Music)
MIDI_FOLDER = "/content/drive/MyDrive/MidiLM/midis"
!python dataset/prepare_data.py --midi_dir {MIDI_FOLDER} --augment

In [ ]:
# 4. Train Tokenizer
!python train_tokenizer.py

In [ ]:
# 5. Training
!python train/train.py --colab --epochs 200 --batch_size 4 --grad_accum 4

In [ ]:
# 6. Export ke GGUF
!python export_gguf.py
import os
if not os.path.exists('llama.cpp/build/bin/llama-quantize'):
    !git clone https://github.com/ggerganov/llama.cpp
    %cd llama.cpp
    !cmake -B build
    !cmake --build build --config Release -j2 --target llama-quantize
    %cd ..
!chmod +x ./llama.cpp/build/bin/llama-quantize
!./llama.cpp/build/bin/llama-quantize checkpoints/midilm-f16.gguf checkpoints/midilm-chat-q4_k_m.gguf Q4_K_M